In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"

%matplotlib inline

# Morphing through the shapes zoo

This tutorial builds an animation that smoothly **morphs** a cloud of black dots from one 3-D shape into the next, with a title that tracks the current shape. It uses a single `hyp.plot(..., animate='morph')` call plus hypertools' own morph *schedule* to drive the title.

HyperTools ships a "shapes zoo" of classic point clouds (`bunny`, `cube`, `sphere`, `vase`, ...), each downloaded once and cached in `~/hypertools_data`, so this notebook is fully offline after the first run.

## 1. Imports

In [2]:
import numpy as np

import hypertools as hyp
from hypertools.plot import morph as _morph

## 2. Load and normalize the shapes

We center each cloud and scale it into the hypertools `[-1, 1]` cube. That rescaling is *not* redundant with what `hyp.plot` does: plot normalizes all datasets with **one shared affine**, so clouds left in their own raw units would be drawn at wildly different sizes.

We also cap each cloud at `N` points by hand instead of leaving it to `morph_samples=N`. The two are equivalent for a plain morph, but here the first cloud is repeated at the end to close the loop and has to be the *same* sample both times, whereas `morph_samples` draws a fresh subset per dataset.

Two of the zoo's shapes need a decision:

* **`teapot` is left out.** `hyp.load('teapot')` returns 1728 rows but only **301 unique coordinates** (ratio 0.174), so sampling rows without replacement yields a few hundred distinct dots and that segment renders as a sparse lattice rather than a solid cloud. The shapes used here are essentially all-unique (bunny 35947/35947, vase 36022/36022, cube 30034/30246, sphere 29891/30135).
* **the `cube` is scaled to 0.8.** Normalized, it reaches ±1 on every axis, which is exactly the drawn axes box, so those frames read as noise inside a wireframe. At 0.8 it sits visibly inside the box, and the other shapes still set the shared box.

In [3]:
def normalize(points):
    points = np.asarray(points, dtype=float)
    points = points - points.mean(axis=0)
    return points / np.abs(points).max()


SHAPES = ['bunny', 'cube', 'sphere', 'teapot', 'vase']
TITLES = ['Bunny', 'Cube', 'Sphere', 'Teapot', 'Vase']
N = 2000
CUBE_SCALE = 0.8
rng = np.random.default_rng(0)


def load(name):
    points = normalize(hyp.load(name))
    if name == 'cube':
        points = points * CUBE_SCALE
    idx = rng.choice(len(points), size=min(N, len(points)), replace=False)
    return points[idx]


clouds = [load(name) for name in SHAPES]
# close the loop: the SAME sampled array, so the closing and opening holds
# draw an identical point set
clouds.append(clouds[0])
titles = TITLES + [TITLES[0]]

## 3. The `hyp.plot` morph call

`animate='morph'` alternates **hold** segments (the camera orbits a finished shape) with **transition** segments (one shape flowing into the next). With the repeated first cloud there are 5 datasets, so the schedule runs `[hold_1, morph_1->2, hold_2, ..., hold_5]`: 9 segments.

The `rotations` list sets each segment's *screen time* (camera speed is constant): a full turn per shape and half a turn per transition, with the first shape's hold split in half across the two ends so the halves play back-to-back on repeat. The total, 6.0, is a whole number of turns, so the camera azimuth wraps exactly at the loop point too.

The one-to-one **point matching** is what makes the morph continuous: every dot is assigned a specific partner in the next shape and travels to it, rather than the cloud reshuffling itself. It is solved as a Hungarian assignment (`scipy.optimize.linear_sum_assignment`), whose cost grows roughly as O(n^3) in the number of points, which is why the clouds are capped first. `morph_samples=N` applies the same cap to any cloud that arrives larger than `N`.

In [4]:
rotations = [0.75] + [0.5, 1.0] * (len(SHAPES) - 1) + [0.5, 0.75]
duration, fps = 12, 20

fig, ani = hyp.plot(clouds, fmt='.', color='k', markersize=1.6,
                    animate='morph', rotations=rotations,
                    morph_samples=N, duration=duration,
                    frame_rate=fps, size=(6, 6), show=False)

## 4. A title that tracks the current shape

We read the current segment for each frame straight out of the same schedule `hyp.plot` uses (`morph_schedule` / `frame_to_segment`), so the label is always in lock-step with what is on screen: we name the shape while *holding* (even segments) and name the pair being morphed on the odd, transition segments.

`azim0=-60` is `hyp.plot`'s default `azim`, and it has to match: the schedule is *recomputed* here rather than read back off the figure, and its per-frame azimuth track accumulates from `azim0`, so a different starting angle would hand back a schedule that no longer tracks the rendered camera. (The title itself only needs `frame_counts`, which does not depend on `azim0`, but keeping the two in step means the camera track stays usable too.)

In [5]:
total_frames = int(round(fps * duration))
frame_counts, _, _ = _morph.morph_schedule(
    len(clouds), total_frames, rotations, azim0=-60)
label = fig.text(0.5, 0.95, '', ha='center', va='top', fontsize=16,
                 fontweight='bold', color='#1a1a1a')


def shape_title(frame):
    seg, _step, _n = _morph.frame_to_segment(frame_counts, frame)
    # transitions (odd segments) show NOTHING, so the label never sits
    # over a half-formed cloud
    return titles[seg // 2] if seg % 2 == 0 else ''


_orig = ani._func


def _wrapped(frame, *args):
    result = _orig(frame, *args)
    label.set_text(shape_title(frame))
    return result


ani._func = _wrapped

## 5. Display (or save) the animation

In [6]:
fig.set_dpi(100)  # halve hypertools' default 200-dpi canvas for a lighter GIF
ani.save('morph_zoo.gif', fps=fps)
print('saved morph_zoo.gif')

saved morph_zoo.gif


![morphing through the shapes zoo](morph_zoo.gif)